In [0]:
from pyspark.sql.functions import first, rand, count
dates = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2024-03-24', INTERVAL 1 DAY)) as  calendar_date")
c_id = spark.sql("SELECT explode(sequence(1,200, 1)) as  client_id")
types = spark.sql("""SELECT concat("col_", colName) as col_name from (SELECT explode(sequence(1,20, 1)) as  colName)""")
 
dates = dates.repartition(99)
c_id = c_id.repartition(11)
types = types.repartition(1)
 
df_cartesian = c_id.crossJoin(dates.select("calendar_date")).crossJoin(types.select("col_name")).select("client_id","calendar_date","col_name")
df_cartesian2 = df_cartesian.groupBy("calendar_date").agg(count("client_id"))
 
# display(df_cartesian2.limit(1000))
 
df_cartesian = df_cartesian.withColumn("val", (rand()*10).cast("int"))

df_grp = df_cartesian.groupBy("client_id","calendar_date").pivot("col_name").agg((first("val").alias("val")))

#display(df_grp)


In [0]:
df = df_grp.limit(1000000)
df2=df_grp.limit(1000000)


In [0]:

df_inner = df.alias("a").join(df2.alias("b"), 
                               ["client_id", "calendar_date"], 
                               "inner")

In [0]:

df_left = df.alias("a").join(df2.alias("b"), 
                              ["client_id", "calendar_date"], 
                              "left")

In [0]:
deduplicated_df_left = df_left.select(
    [col("a." + c).alias(c) for c in df.columns]
)
display(deduplicated_df_left.limit(5))

client_id,calendar_date,col_1,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_2,col_20,col_3,col_4,col_5,col_6,col_7,col_8,col_9
170,2024-03-19,7,1,2,5,3,5,7,4,7,3,7,1,3,3,0,3,5,0,8,5
170,2024-03-07,6,1,9,0,6,8,4,3,6,8,7,3,9,1,5,5,7,7,7,3
170,2024-01-17,7,3,3,1,6,6,3,4,0,5,7,9,6,7,9,2,1,5,1,2
170,2024-01-31,3,1,6,7,5,3,8,4,8,6,0,5,3,4,5,8,5,4,3,1
170,2024-02-26,1,4,1,3,6,6,7,9,6,8,1,7,1,7,7,1,0,6,3,8


In [0]:
deduplicated_df_inner = df_inner.select(
    [col("a." + c).alias(c) for c in df.columns]
)
display(deduplicated_df_inner.limit(5))

client_id,calendar_date,col_1,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_2,col_20,col_3,col_4,col_5,col_6,col_7,col_8,col_9
170,2024-03-19,7,1,2,5,3,5,7,4,7,3,7,1,3,3,0,3,5,0,8,5
170,2024-03-07,6,1,9,0,6,8,4,3,6,8,7,3,9,1,5,5,7,7,7,3
170,2024-01-17,7,3,3,1,6,6,3,4,0,5,7,9,6,7,9,2,1,5,1,2
170,2024-01-31,3,1,6,7,5,3,8,4,8,6,0,5,3,4,5,8,5,4,3,1
170,2024-02-26,1,4,1,3,6,6,7,9,6,8,1,7,1,7,7,1,0,6,3,8
